# Création d'une GWAS et d'un PRS sur la population AFR

In [1]:
import os
import pandas as pd
import seaborn as sns
import subprocess
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split

## Téléchargement des données génomique AOU

In [1]:
!mkdir -p GWAS_AFR

In [2]:
!gsutil -u $GOOGLE_PROJECT -m cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.* GWAS_AFR/

Copying gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed...
Copying gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim...       
Copying gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam...       
\ [3/3 files][181.2 GiB/181.2 GiB] 100% Done  61.4 MiB/s ETA 00:00:00           
Operation completed over 3 objects/181.2 GiB.                                    


## Importation des données

In [2]:
name_of_file_in_bucket = "df_final_cohort.tsv"

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
df = pd.read_csv(name_of_file_in_bucket)
df.head()

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_final_cohort.tsv...
| [1 files][117.0 MiB/117.0 MiB]                                                
Operation completed over 1 objects/117.0 MiB.                                    


[INFO] df_final_cohort.tsv is successfully downloaded into your working space


,B,race,ethnicity,age,self_reported_category,has_BC,eur_rye,eas_rye,amr_rye,afr_rye,...,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000039,Black or African American,Not Hispanic or Latino,54,Black or African American,0,0.045004,0.003594,0.000000,0.874350,...,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
1,1000045,Asian,Not Hispanic or Latino,67,Asian,0,0.000000,0.939365,0.000000,0.007806,...,0.001778,0.000698,-0.000009,0.000727,-0.000209,0.000062,-0.000256,-0.000293,-0.000274,-0.000426
2,1000059,White,Not Hispanic or Latino,68,White,0,0.781992,0.000000,0.120046,0.000000,...,-0.010278,-0.001716,-0.000227,0.000956,0.000586,-0.001687,0.000547,-0.001534,0.001173,-0.000093
3,1000062,White,Not Hispanic or Latino,66,White,0,0.783972,0.000952,0.091612,0.000000,...,-0.011289,-0.000806,-0.000868,0.000160,0.000223,-0.000703,0.000848,-0.000522,0.000670,0.000393
4,1000070,White,Not Hispanic or Latino,78,White,0,0.894111,0.000000,0.050451,0.000000,...,-0.017076,-0.002684,-0.002102,0.000015,0.000914,-0.000492,-0.000415,-0.001789,0.001531,-0.000028


## Création des jeu de données d'entraînement et de test

In [4]:
# Diviser le DataFrame en un ensemble d'entraînement (95%) et de test (5%) en conservant la proportion de la variable cible 'has_BC'
train_df, test_df = train_test_split(
    df,
    test_size=0.01,
    stratify=df["has_BC"],  # Maintient la proportion de classes dans chaque ensemble
    random_state=42         # Pour assurer la reproductibilité
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 264492, Test size: 2672


In [5]:
print("Train proportions :")
print(train_df["has_BC"].value_counts(normalize=True))

print("\nTest proportions :")
print(test_df["has_BC"].value_counts(normalize=True))

Train proportions :
has_BC
0    0.961182
1    0.038818
Name: proportion, dtype: float64

Test proportions :
has_BC
0    0.961078
1    0.038922
Name: proportion, dtype: float64


### Enregistrement des données `train` sur le bucket

In [6]:
# Enregistre le DataFrame final dans un fichier TSV local
destination_filename = 'train_df_final_cohort.tsv'
train_df.to_csv(destination_filename, index=False)

# Récupère le nom du bucket de l'environnement d'exécution
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local vers le dossier "Data" du bucket Google Cloud
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuels messages d’erreur de la commande gsutil
output.stderr

b'Copying file://./train_df_final_cohort.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/115.8 MiB]                                                \r-\r- [0 files][  2.6 MiB/115.8 MiB]                                                \r\\\r|\r| [0 files][  4.6 MiB/115.8 MiB]                                                \r/\r/ [0 files][  6.4 MiB/115.8 MiB]                                                \r-\r- [0 files][  8.2 MiB/115.8 MiB]                                                \r\\\r|\r| [0 files][ 10.0 MiB/115.8 MiB]                                                \r/\r/ [0 files][ 11.9 MiB/115.8 MiB]                                                \r-\r\\\r\\ [0 files][ 13.7 MiB/115.8 MiB]                                                \r|\r| [0 files][ 15.5 MiB/115.8 MiB]                                                \r/\r/ [0 files][ 17.3 MiB/115.8 MiB]                                                \r-\r\\\r\\ [0 files][ 19.1 MiB/115.8 MiB]          

### Enregistrement des données `test` sur le bucket

In [7]:
# Enregistre le DataFrame final dans un fichier TSV local
destination_filename = 'test_df_final_cohort.tsv'
test_df.to_csv(destination_filename, index=False)

# Récupère le nom du bucket de l'environnement d'exécution
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local vers le dossier "Data" du bucket Google Cloud
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuels messages d’erreur de la commande gsutil
output.stderr

b'Copying file://./test_df_final_cohort.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/  1.2 MiB]                                                \r/ [1 files][  1.2 MiB/  1.2 MiB]                                                \r-\r\nOperation completed over 1 objects/1.2 MiB.                                      \n'

## Sélection des individus AFR (score RYE > 0.8)

In [8]:
threshold = 0.8
category_cols = ['eur_rye', 'eas_rye', 'amr_rye', 'afr_rye', 'sas_rye', 'mid_rye']

count_per_category = (df[category_cols] > threshold).sum()
count_per_category

eur_rye    111399
eas_rye      5767
amr_rye     29430
afr_rye     31866
sas_rye      1789
mid_rye       147
dtype: int64

In [9]:
df

,B,race,ethnicity,age,self_reported_category,has_BC,eur_rye,eas_rye,amr_rye,afr_rye,...,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000039,Black or African American,Not Hispanic or Latino,54,Black or African American,0,0.045004,0.003594,0.000000,0.874350,...,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
1,1000045,Asian,Not Hispanic or Latino,67,Asian,0,0.000000,0.939365,0.000000,0.007806,...,0.001778,0.000698,-0.000009,0.000727,-0.000209,0.000062,-0.000256,-0.000293,-0.000274,-0.000426
2,1000059,White,Not Hispanic or Latino,68,White,0,0.781992,0.000000,0.120046,0.000000,...,-0.010278,-0.001716,-0.000227,0.000956,0.000586,-0.001687,0.000547,-0.001534,0.001173,-0.000093
3,1000062,White,Not Hispanic or Latino,66,White,0,0.783972,0.000952,0.091612,0.000000,...,-0.011289,-0.000806,-0.000868,0.000160,0.000223,-0.000703,0.000848,-0.000522,0.000670,0.000393
4,1000070,White,Not Hispanic or Latino,78,White,0,0.894111,0.000000,0.050451,0.000000,...,-0.017076,-0.002684,-0.002102,0.000015,0.000914,-0.000492,-0.000415,-0.001789,0.001531,-0.000028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267159,9999373,None of these,What Race Ethnicity: Race Ethnicity None Of These,53,None of these,0,0.987243,0.000000,0.000000,0.012757,...,-0.019134,-0.004776,-0.001668,-0.001610,0.000282,-0.000176,-0.000780,-0.001432,-0.001350,-0.000589
267160,9999653,Black or African American,Hispanic or Latino,43,More than one population,0,0.070018,0.031847,0.459473,0.419679,...,-0.007309,0.007351,-0.003693,-0.002859,-0.003085,-0.000890,0.006171,0.006674,0.003420,0.004854
267161,9999678,White,Not Hispanic or Latino,61,White,0,0.913647,0.000000,0.028156,0.000000,...,-0.017846,-0.002502,-0.000634,0.000194,0.000252,-0.001290,0.001715,-0.000752,-0.000415,-0.000020
267162,9999697,None Indicated,Hispanic or Latino,39,What Race Ethnicity: Hispanic,0,0.000000,0.000000,0.947722,0.000000,...,0.010985,0.002458,0.001513,-0.001535,-0.001271,-0.001320,-0.000950,-0.001408,-0.003327,0.000309


In [10]:
df_afr_rye = df[df['afr_rye']>threshold]#['B']
df_afr_rye

,B,race,ethnicity,age,self_reported_category,has_BC,eur_rye,eas_rye,amr_rye,afr_rye,...,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000039,Black or African American,Not Hispanic or Latino,54,Black or African American,0,0.045004,0.003594,0.0,0.874350,...,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
13,1000151,Black or African American,Not Hispanic or Latino,61,Black or African American,0,0.098715,0.000000,0.0,0.856808,...,-0.007028,-0.004291,-0.002965,-0.005485,0.000885,-0.001218,-0.000295,0.004021,-0.003528,-0.002400
19,1000195,Black or African American,Not Hispanic or Latino,43,Black or African American,0,0.000000,0.000000,0.0,1.000000,...,0.006064,0.016884,0.014196,-0.002095,0.004388,0.005839,-0.002770,0.006232,-0.003948,-0.003948
21,1000204,Black or African American,Not Hispanic or Latino,42,Black or African American,0,0.028545,0.000379,0.0,0.894432,...,-0.012882,0.010865,0.000213,-0.005382,-0.005989,0.000485,0.003581,0.003233,-0.000076,-0.000995
26,1000265,Black or African American,Not Hispanic or Latino,56,Black or African American,0,0.096582,0.001398,0.0,0.819030,...,-0.009190,0.004858,-0.003130,-0.002254,0.001146,0.001233,0.005653,-0.006717,0.002847,-0.005516
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267089,9993055,Black or African American,Not Hispanic or Latino,48,Black or African American,0,0.074485,0.001683,0.0,0.859096,...,-0.016912,-0.005496,0.000945,-0.002369,0.007921,0.004988,-0.002738,-0.009650,0.002564,0.009263
267094,9993483,Black or African American,Not Hispanic or Latino,63,Black or African American,0,0.000000,0.000000,0.0,0.980956,...,-0.011759,0.014860,0.015780,0.004658,0.006191,0.000011,0.005454,0.009308,0.006102,-0.005828
267108,9994528,Black or African American,Not Hispanic or Latino,29,Black or African American,0,0.000000,0.000000,0.0,0.937690,...,-0.021120,0.014399,0.018393,-0.000326,0.007429,0.000341,0.004609,0.008339,0.004847,0.007533
267136,9997625,Black or African American,Not Hispanic or Latino,46,Black or African American,0,0.049284,0.000000,0.0,0.917373,...,-0.006881,0.003479,0.005549,-0.003026,0.003003,0.002809,-0.004467,-0.003431,-0.012149,-0.007805


In [27]:
df_afr_rye['B'].to_csv('GWAS_AFR/afr_id.csv',index=False,header=False)

## Sélection des individus 

In [9]:
!mkdir -p GWAS_AFR/results

In [10]:
%%bash

plink2 --bfile GWAS_AFR/arrays \
      --keep GWAS_AFR/afr_id.csv \
      --chr 21 \
      --make-pgen \
      --threads 16 \
      --out GWAS_AFR/chr21_eur

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to GWAS_AFR/chr21_eur.log.
Options in effect:
  --bfile GWAS_AFR/arrays
  --chr 21
  --keep GWAS_AFR/afr_id.csv
  --make-pgen
  --out GWAS_AFR/chr21_eur
  --threads 16

Start time: Mon Jun  2 15:50:13 2025
7141 MiB RAM detected, ~5543 available; reserving 3570 MiB for main workspace.
Using up to 16 threads (change this with --threads).
447278 samples (0 females, 0 males, 447278 ambiguous; 447278 founders) loaded
from GWAS_AFR/arrays.fam.
22712 out of 1739269 variants loaded from GWAS_AFR/arrays.bim.
Note: No phenotype data present.
--keep: 31866 samples remaining.
31866 samples (0 females, 0 males, 31866 ambiguous; 31866 founders) remaining
after main filters.
Writing GWAS_AFR/chr21_eur.psam ... done.
Writing GWAS_AFR/chr21_eur.pvar ... 101011111212131314141515161617171818191920202121222223232425252626272728282929303031313

## Application de la GWAS

In [19]:
df_pheno = df_afr_rye[['B','has_BC']].rename(columns={'B':'FID'})

In [22]:
# Dupliquer la colonne FID en tant que IID
df_pheno["IID"] = df_pheno["FID"]

In [26]:
df_pheno = df_pheno[['FID', 'IID', 'has_BC']]
df_pheno

,FID,IID,has_BC
0,1000039,1000039,0
13,1000151,1000151,0
19,1000195,1000195,0
21,1000204,1000204,0
26,1000265,1000265,0
...,...,...,...
267089,9993055,9993055,0
267094,9993483,9993483,0
267108,9994528,9994528,0
267136,9997625,9997625,0


In [28]:
df_pheno.to_csv("GWAS_AFR/phenotype.txt", sep='\t', index=False)

In [53]:
df_pheno

,FID,IID,has_BC
0,1000039,1000039,0
13,1000151,1000151,0
19,1000195,1000195,0
21,1000204,1000204,0
26,1000265,1000265,0
...,...,...,...
267089,9993055,9993055,0
267094,9993483,9993483,0
267108,9994528,9994528,0
267136,9997625,9997625,0


In [33]:
df_afr_rye = df_afr_rye.rename(columns={'B':'FID'})

In [34]:
# Dupliquer la colonne FID en tant que IID
df_afr_rye["IID"] = df_afr_rye["FID"]

In [40]:
df_covar = df_afr_rye[['FID','IID','age','PC1','PC2','PC3','PC4','PC5','PC6','PC7','PC8','PC9','PC10']]
df_covar

,FID,IID,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,1000039,1000039,54,-0.267138,0.004698,0.001046,0.001767,0.031375,0.001713,-0.009450,0.016385,-0.000844,0.001511
13,1000151,1000151,61,-0.259287,0.007598,-0.001849,0.008709,0.004493,-0.000182,-0.007028,-0.004291,-0.002965,-0.005485
19,1000195,1000195,43,-0.317186,-0.010157,-0.001107,0.000364,0.021758,0.012732,0.006064,0.016884,0.014196,-0.002095
21,1000204,1000204,42,-0.276070,0.001328,-0.001371,0.004225,0.008361,-0.007882,-0.012882,0.010865,0.000213,-0.005382
26,1000265,1000265,56,-0.244960,0.012532,0.001441,0.008956,0.019534,-0.005273,-0.009190,0.004858,-0.003130,-0.002254
...,...,...,...,...,...,...,...,...,...,...,...,...,...
267089,9993055,9993055,48,-0.262167,0.003949,-0.005142,0.005009,0.002770,-0.008204,-0.016912,-0.005496,0.000945,-0.002369
267094,9993483,9993483,63,-0.308654,-0.006375,0.002200,0.005098,0.006581,-0.004569,-0.011759,0.014860,0.015780,0.004658
267108,9994528,9994528,29,-0.293129,-0.003982,-0.000025,0.002801,0.013716,-0.012649,-0.021120,0.014399,0.018393,-0.000326
267136,9997625,9997625,46,-0.283435,0.006296,-0.002731,0.005815,0.010914,0.000798,-0.006881,0.003479,0.005549,-0.003026


In [41]:
df_covar.to_csv('GWAS_AFR/covariables.txt', sep='\t', index=False)

In [52]:
%%bash

plink2 --pfile GWAS_AFR/chr21_eur \
       --pheno GWAS_AFR/phenotype.txt \
       --pheno-name has_BC \
       --covar GWAS_AFR/covariables.txt \
       --covar-name age PC1 PC2 PC3 PC4 PC5 PC6 PC7 PC8 PC9 PC10 \
       --glm \
       --out GWAS_AFR/gwas_chr21

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to GWAS_AFR/gwas_chr21.log.
Options in effect:
  --covar GWAS_AFR/covariables.txt
  --covar-name age PC1 PC2 PC3 PC4 PC5 PC6 PC7 PC8 PC9 PC10
  --glm
  --out GWAS_AFR/gwas_chr21
  --pfile GWAS_AFR/chr21_eur
  --pheno GWAS_AFR/phenotype.txt
  --pheno-name has_BC

Start time: Tue Jun  3 23:36:20 2025
7141 MiB RAM detected, ~5598 available; reserving 3570 MiB for main workspace.
Using up to 8 compute threads.
31866 samples (0 females, 0 males, 31866 ambiguous; 31866 founders) loaded from
GWAS_AFR/chr21_eur.psam.
22712 variants loaded from GWAS_AFR/chr21_eur.pvar.


Error: No entries in GWAS_AFR/phenotype.txt correspond to loaded sample IDs.


End time: Tue Jun  3 23:36:20 2025


CalledProcessError: Command 'b'\nplink2 --pfile GWAS_AFR/chr21_eur \\\n       --pheno GWAS_AFR/phenotype.txt \\\n       --pheno-name has_BC \\\n       --covar GWAS_AFR/covariables.txt \\\n       --covar-name age PC1 PC2 PC3 PC4 PC5 PC6 PC7 PC8 PC9 PC10 \\\n       --glm \\\n       --out GWAS_AFR/gwas_chr21\n'' returned non-zero exit status 7.

## Création du PRS

In [ ]:
plink2 --pfile chr21_target \
       --score prs_weights.txt 1 2 3 \
       --out prs_scores